# Rasch + 2PL Workflow (mirt-style)

This notebook mirrors the **R `mirt`** workflow using the LSAT7 dataset.

In R, a typical mirt workflow looks like:

```r
library(mirt)
data <- expand.table(LSAT7)
mod_rasch <- mirt(data, 1, itemtype = "Rasch")
mod_2pl <- mirt(data, 1)
coef(mod_rasch, simplify=TRUE, IRTpars=TRUE)
fscores(mod_rasch, method="EAP")
```

We will do the equivalent steps in Python and compare to reference outputs.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from irt import fit

## 1) Load the LSAT7 data (same data used by mirt)

This file comes from the R `mirt` package reference outputs in `tests/mirt_reference/`.

In [ ]:
lsat7 = pd.read_csv("../tests/mirt_reference/lsat7_data.csv")
lsat7.shape

## 2) Fit Rasch (mirt-style)

`mirt` uses 61 quadrature points by default, which matches this library's default settings. So we can use the defaults here.

In [ ]:
rasch_result = fit(lsat7, model="rasch", estimator="mml_em")
rasch_result

## 3) Compare Rasch parameters and log-likelihood to mirt

We compare **centered** difficulties and log-likelihood. Small differences are normal because of numeric integration details.

In [ ]:
rasch_ref = pd.read_csv("../tests/mirt_reference/lsat7_rasch_params.csv")

b_py = rasch_result.params["b"]
b_ref = rasch_ref["b"].values

b_py_centered = b_py - b_py.mean()
b_ref_centered = b_ref - b_ref.mean()

corr = np.corrcoef(b_py_centered, b_ref_centered)[0, 1]
mad = np.abs(b_py_centered - b_ref_centered).mean()

with open("../tests/mirt_reference/lsat7_rasch_loglik.txt") as f:
    loglik_ref = float(f.read().strip())

loglik_diff = rasch_result.loglik - loglik_ref

corr, mad, loglik_diff

## 4) EAP and MAP scores (compare to mirt)

`mirt` provides EAP and MAP scores via `fscores()`. We'll compare correlations.

In [ ]:
scores_eap = rasch_result.score(method="eap")
scores_map = rasch_result.score(method="map")

mirt_eap = pd.read_csv("../tests/mirt_reference/lsat7_rasch_eap.csv")
mirt_map = pd.read_csv("../tests/mirt_reference/lsat7_rasch_map.csv")

eap_corr = np.corrcoef(scores_eap.theta, mirt_eap["theta"].values)[0, 1]
map_corr = np.corrcoef(scores_map.theta, mirt_map["theta"].values)[0, 1]

eap_corr, map_corr

## 5) Fit 2PL (mirt-style)

In `mirt`, 2PL is the default item type. Here we set `model="2pl"`.

In [ ]:
twopl_result = fit(lsat7, model="2pl", estimator="mml_em")
twopl_result

## 6) Compare 2PL parameters to mirt

We compare discriminations (`a`) directly and difficulties (`b`) after centering.

In [ ]:
twopl_ref = pd.read_csv("../tests/mirt_reference/lsat7_2pl_params.csv")

# Discriminations (a)
a_corr = np.corrcoef(twopl_result.params["a"], twopl_ref["a"].values)[0, 1]

# Difficulties (b) after centering
b_py = twopl_result.params["b"] - twopl_result.params["b"].mean()
b_ref = twopl_ref["b"].values - twopl_ref["b"].values.mean()
b_mad = np.abs(b_py - b_ref).mean()

with open("../tests/mirt_reference/lsat7_2pl_loglik.txt") as f:
    loglik_ref = float(f.read().strip())

loglik_diff = twopl_result.loglik - loglik_ref

a_corr, b_mad, loglik_diff

## 7) 2PL scoring

Scoring works the same way as Rasch. You can compute EAP or MAP scores for 2PL as well.

In [ ]:
twopl_scores = twopl_result.score(method="eap")
twopl_scores.to_dataframe().head()

## What you learned

- How to reproduce the **mirt Rasch** and **mirt 2PL** workflows in Python
- How to compare item parameters and log-likelihoods with R reference values
- How to score abilities with EAP/MAP for both models